# Disease-balanced cap 재학습 (Colab)

**푸시 없이** 로컬에서 수정한 파일만 업로드해서 실행합니다.

순서: ① 패치 업로드 → ② cap 분포 확인 → ③ 학습 → ④ 5k vs balanced 비교

필요: Google Drive에 `eye_data/` + (선택) 기존 `dog_best_random_split.pth`

In [ ]:
# ── 설정 (본인 경로에 맞게 수정) ──────────────────────────────────
ANIMAL_TYPE = "dog"          # dog | cat
SPLIT_SEED = 42
DISEASE_BALANCED_LIMIT = 5000
BATCH_SIZE = 32              # OOM 시 16
PHASE1_EPOCHS = 4
PHASE2_EPOCHS = 12

# Drive에 복사해 둔 프로젝트 루트 (eye_data/, models/ 포함)
REPO_ROOT = "/content/drive/MyDrive/capstone_petcare"

# True: 아래 셀에서 수정 파일 5개 업로드 (git push 불필요)
UPLOAD_PATCH = True

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print("cwd:", os.getcwd())
print("eye_data exists:", os.path.isdir("eye_data"))

In [ ]:
!pip -q install timm albumentations opencv-python-headless scikit-learn tqdm

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# ── 로컬 수정 파일 업로드 (푸시 대신) ─────────────────────────────
# 맥/PC에서 아래 5개 파일을 선택 업로드:
#   models/classifier/dataset_random_split.py
#   models/classifier/train_random_split.py
#   models/classifier/analyze_cap_distribution.py
#   models/classifier/compare_cap_models.py
#   models/classifier/train.py

import shutil
from pathlib import Path
from google.colab import files

PATCH_DIR = Path("models/classifier")

if UPLOAD_PATCH:
    uploaded = files.upload()  # 5개 py 파일 multi-select
    for name, data in uploaded.items():
        dest = PATCH_DIR / Path(name).name
        dest.write_bytes(data)
        print(f"✓ patched {dest}")
else:
    print("UPLOAD_PATCH=False — Drive에 이미 반영된 파일 사용")

In [ ]:
import os

def env(**kwargs):
    for k, v in kwargs.items():
        os.environ[k] = str(v)

env(
    ANIMAL_TYPE=ANIMAL_TYPE,
    SPLIT_SEED=SPLIT_SEED,
    VAL_RATIO="0.2",
    USE_GROUP_SPLIT="1",
    BATCH_SIZE=BATCH_SIZE,
    NUM_WORKERS="2",
    IMG_SIZE="300",
    PHASE1_EPOCHS=PHASE1_EPOCHS,
    PHASE2_EPOCHS=PHASE2_EPOCHS,
)

## 1) Cap 분포 확인
백내장:핵경화 비정상 비율이 **~1:1** 인지 확인 후 학습 진행

In [ ]:
# balanced cap
env(
    CAP_MODE="disease_balanced",
    DISEASE_BALANCED_LIMIT=DISEASE_BALANCED_LIMIT,
    PRESERVE_SMARTPHONE="true",
)
!python models/classifier/analyze_cap_distribution.py --animal {ANIMAL_TYPE}

In [ ]:
# (선택) 기존 5k stratum cap 분포 비교
env(CAP_MODE="stratum", MAX_PER_CLASS="5000", DISEASE_CAPS="")
!python models/classifier/analyze_cap_distribution.py --animal {ANIMAL_TYPE}

## 2) 학습 → `dog_best_balanced_cap.pth`

In [ ]:
env(
    CAP_MODE="disease_balanced",
    DISEASE_BALANCED_LIMIT=DISEASE_BALANCED_LIMIT,
    PRESERVE_SMARTPHONE="true",
    RESUME_CHECKPOINT="0",  # 처음부터; 이어학습 시 auto
)
!python models/classifier/train_random_split.py

## 3) 5k cap vs balanced cap 비교
기존 `dog_best_random_split.pth`가 Drive checkpoints에 있어야 합니다.

In [ ]:
!python models/classifier/compare_cap_models.py --animal {ANIMAL_TYPE}

In [ ]:
# 체크포인트 Drive에 백업 (이미 REPO_ROOT 안에 저장됨)
from pathlib import Path
ckpt = Path(f"models/classifier/checkpoints/{ANIMAL_TYPE}_best_balanced_cap.pth")
print("exists:", ckpt.is_file(), ckpt)
if ckpt.is_file():
    from google.colab import files
    files.download(str(ckpt))